# 决策树与集成学习：咖啡店怎样准备面包？

![决策树示意：先判断广告投入是否至少为 5，再根据需要判断是否有活动，沿分支到达售罄或未售罄的预测](../assets/tree_decision_diagram.png)

咖啡店想预测当天面包是否会售罄。上图是一棵**为讲解构造的示意树**：从顶部开始，每个浅色框提出一个问题；根据答案沿一条分支向下走，直到到达彩色的**叶节点**，得到预测。

例如，广告投入为 3（单位：百元），当天有活动：第一个问题的答案是「否」，接着「有活动？」的答案是「是」，于是走到「预测：售罄」。如果广告投入为 6，则第一个问题回答「是」后就直接到达叶节点，不需要再问活动。这是预测而非保证；图中的规则只是示意，不是下面数据训练出的树。

先猜一猜：如果一直追问，直到记住训练中的每一天，对从未见过的新一天一定更准吗？多棵树又能怎样互相帮助？本课数据是模拟的，只用于学习。按顺序运行单元格；「演示」使用滑块，「练习」可以修改代码后重新运行后续单元格。


## 1. 训练一棵树：先比较哪一个问题更好

![六条样本的决策树划分示意：比较两个候选门槛，将不纯度按左右样本数加权，再选择较纯的划分](../assets/tree_split_mechanics.png)

先看图中**独立的六条简化记录**，广告投入从 1 到 6，前三天未售罄，后三天售罄。假设我们要从两个问题中选择一个：广告投入不超过 2.5，还是不超过 3.5？后者把两类样本完全分开；前者右边仍混着两种结果。

一边只包含同一种结果，就称这一边是**纯**的；如果混着多类，就有不纯度。比较两个问题时，不能只看某一边：要把左右两边的不纯度按各自样本数加权。图中先算出两个候选的结果，再选择更小的那个。实际训练会在当前节点比较多个特征和门槛；选定问题后，每个分支还可以继续分裂。

**训练和预测是两件事**：训练时用已知标签比较问题、确定树结构；预测新样本时，只用它的特征沿着已学好的问题走到叶子，不使用它的真实标签。接下来我们把图中的“不纯”写成可计算的数字。

### 用基尼不纯度衡量混杂

分类树常用**基尼不纯度**衡量一个节点里的类别有多混杂：

$$G=1-\sum_k p_k^2,\qquad G_{split}=\frac{n_L}{n}G_L+\frac{n_R}{n}G_R$$

$p_k$ 是节点中第 $k$ 类的比例，$n_L$、$n_R$ 是两边的样本数。纯节点 $G=0$；二分类各占一半时 $G=0.5$。比较同一节点的候选问题时，加权不纯度越小，下降越多，划分就越纯；这是一种在当前节点做出的贪心选择，并不保证整棵树全局最优。

下面换成**另一组 10 天数据**，试问「宣传投入 ≤ 4.5 吗？」左右各 5 天：左边售罄 1 天，右边售罄 4 天，两边基尼都是 $0.32$，加权后仍为 $0.32$；原节点两类各 5 天，基尼为 $0.5$，因此下降 $0.18$。这只是一个候选门槛，不一定是最优的。先运行下格计算，再用滑块亲自比较。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_squared_error

# 为了看清一次划分，先只看无活动日的宣传与售罄记录。
ad = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=float)
sold_out = np.array([0, 0, 0, 1, 0, 1, 1, 0, 1, 1])
plt.figure(figsize=(7, 2.8))
plt.scatter(ad, sold_out, s=80, color='#4169a1')
plt.yticks([0, 1], ['No', 'Yes']); plt.xlabel('Advertising (hundreds)')
plt.title('Ten example days'); plt.grid(alpha=.2); plt.show()

**自测准备**：运行下一格以启用各阶段的互动题，不需要修改其中的控件代码。之后可以选择作答或跳过任意一道题；答题不会改变模型或数据。

In [ ]:
# 【自测准备】用于后续单选题；只需运行，无需修改。
# 每次提交只更新本题反馈，不影响模型、数据或其他题目。
from html import escape as _quiz_escape

_quiz_bank = {
    'split': {
        'question': '同一个父节点的两个候选划分，加权基尼分别为 0.25 和 0.10。若只按这个标准选择，应选哪一个？',
        'options': [
            ('A. 选 0.25，越混杂越好', '基尼衡量混杂，不纯度越大并不代表划分越纯。'),
            ('B. 选 0.10，整体更纯', '同一个父节点下，0.10 更小，意味着加权后的不纯度下降更多。'),
            ('C. 两个数无法比较，必须只看左边', '要综合左右子节点，并按各自样本数加权；这里给出的正是加权后的结果。'),
        ], 'correct': 1,
    },
    'overfit': {
        'question': '树越长越深时，训练准确率继续上升，验证准确率却开始下降。这提示什么？',
        'options': [
            ('A. 可能开始拟合训练集噪声，应考虑限制树的复杂度', '训练更好但验证变差是过拟合的常见信号；可以比较深度或剪枝方案。'),
            ('B. 验证集下降证明深树对所有新数据都更差', '一次划分存在波动，验证下降是一个信号，不足以证明所有新数据都会更差。'),
            ('C. 训练准确率高说明验证集不再需要', '训练表现不能替代独立数据上的表现，验证集仍用于模型选择。'),
        ], 'correct': 0,
    },
    'forest': {
        'question': '随机森林怎样让树产生差异，并组合它们的结果？',
        'options': [
            ('A. 各树有放回抽样、随机选候选特征，再聚合预测', '不同重采样和候选特征使树产生差异；再聚合预测有助于降低波动。'),
            ('B. 每棵树只学习前一棵树的残差', '这是平方误差回归中梯度提升的直观做法；森林中的树不依赖前一棵。'),
            ('C. 每棵树用完全相同的训练数据和全部特征，不引入随机性', '这样更容易得到相似的树，失去随机森林通过多样性降低波动的机制。'),
        ], 'correct': 0,
    },
    'boost': {
        'question': '平方误差回归中，当前预测为 7，新树输出 4，学习率为 0.3。更新后的预测是多少？',
        'options': [
            ('A. 11', '先将树的输出乘学习率，再加到原预测；不能直接把 4 全部加上。'),
            ('B. 8.2', '更新是 7 + 0.3 × 4 = 8.2；新树贡献的是修正量。'),
            ('C. 1.2', '0.3 × 4 = 1.2 只是修正量，还需要加上已有预测 7。'),
        ], 'correct': 1,
    },
    'xgb': {
        'question': '只考虑 L2 正则，固定非零 G 和正 H。增大 λ 时，叶权 w* = −G / (H + λ) 的绝对值如何变化？',
        'options': [
            ('A. 变大', '分母 H + λ 随 λ 增大而增大，固定非零分子时比值的绝对值会减小。'),
            ('B. 变小', 'λ 增大使分母增大，固定非零 G 时叶权绝对值缩小；若 G 为 0，叶权一直为 0。'),
            ('C. 一定变为负数', '叶权的符号由 −G 决定，增大 λ 不会改变 G 的符号。'),
        ], 'correct': 1,
    },
    'evaluation': {
        'question': '比较几个模型并调参后，独立测试集应怎样使用？',
        'options': [
            ('A. 每试一组参数都用测试集选最好的结果', '反复依据测试结果选参数会让测试集参与选择，无法再当作独立评估。'),
            ('B. 在验证集选定方案后，用测试集做最终评估', '验证集用于选择，独立测试集用于评估选定方案在未参与选择的数据上的表现。'),
            ('C. 只看训练集，忽略验证和测试', '训练集参与了拟合，不能单独衡量模型对未见数据的表现。'),
        ], 'correct': 1,
    },
}

def lesson_quiz(key):
    """构造一道可重复作答的单选自测，并返回控件。"""
    item = _quiz_bank[key]
    prompt = widgets.HTML(value='<b>' + _quiz_escape(item['question']) + '</b>')
    choices = widgets.RadioButtons(
        options=[(text, i) for i, (text, _) in enumerate(item['options'])],
        value=None, layout=widgets.Layout(width='100%'))
    submit = widgets.Button(description='提交答案', button_style='primary')
    feedback = widgets.HTML(value='')

    def clear_feedback(change):
        feedback.value = ''

    def check_answer(_):
        selected = choices.value
        if selected is None:
            feedback.value = '<span style="color:#8a5a00">请先选择一个答案，再提交。</span>'
            return
        correct = selected == item['correct']
        result = '回答正确。' if correct else '再想一想。'
        color = '#176b4b' if correct else '#a64232'
        explanation = _quiz_escape(item['options'][selected][1])
        feedback.value = f'<span style="color:{color}"><b>{result}</b> {explanation}</span>'

    choices.observe(clear_feedback, names='value')
    submit.on_click(check_answer)
    return widgets.VBox([prompt, choices, submit, feedback], layout=widgets.Layout(width='100%'))

In [ ]:
# 【算法练习】计算基尼不纯度和一次划分后的加权不纯度。
def gini(labels):
    labels = np.asarray(labels)
    if len(labels) == 0:
        raise ValueError('不能计算空节点的不纯度')
    _, counts = np.unique(labels, return_counts=True)
    proportions = counts / counts.sum()
    return 1 - np.sum(proportions ** 2)

def split_gini(values, labels, threshold):
    labels = np.asarray(labels)
    left = np.asarray(values) <= threshold
    if not left.any() or left.all():
        raise ValueError('门槛必须将样本分成非空的两边')
    return left.mean() * gini(labels[left]) + (~left).mean() * gini(labels[~left])

assert np.isclose(gini(np.array([0, 0, 1, 1])), .5)
assert np.isclose(gini(np.array([1, 1])), 0)
assert np.isclose(split_gini(ad, sold_out, 4.5), .32)
print('原节点：', round(gini(sold_out), 3), '；门槛 4.5：', round(split_gini(ad, sold_out, 4.5), 3))

### 拖动门槛，看两边有多纯

每条虚线都是一个候选问题，左边包含等于门槛的样本。先猜哪个门槛最好，再拖动滑块比较加权基尼。下面的演示调用你刚写的 `split_gini`；修改函数后重新运行上面和本单元格。

In [ ]:
# 【演示】直接运行；滑块选取相邻数据点之间的门槛。
def show_split(threshold):
    left = ad <= threshold
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.scatter(ad[left], sold_out[left], s=80, color='#4169a1', label='Left')
    ax.scatter(ad[~left], sold_out[~left], s=80, color='#e88b36', label='Right')
    ax.axvline(threshold, color='#d33f49', linestyle='--')
    ax.set(yticks=[0, 1], yticklabels=['No', 'Yes'], xlabel='Advertising (hundreds)',
           title=f'Weighted Gini = {split_gini(ad, sold_out, threshold):.3f}')
    ax.legend(); ax.grid(alpha=.2); plt.show()

display(widgets.interactive(show_split, threshold=widgets.SelectionSlider(
    options=list(np.arange(.5, 9, 1)), value=4.5, description='Threshold', continuous_update=False)))

### 自测 1：选择划分

单选。运行下面的单元格，选择后点击「提交答案」；可以修改选择并再次提交。自测可跳过作答，不影响后续课程。

In [ ]:
# 【自测】选择后提交，可修改答案重试。
display(lesson_quiz('split'))

## 2. 一棵训练好的分类树怎样预测？

我们已经知道怎样在一个节点选问题。现在用更多模拟日期和三个特征——宣传、活动、天气——让算法连续选择问题，长成一棵树。

先把数据分为**训练集、验证集和测试集**：训练集用来学树，验证集用于本章的观察与模型选择，测试集留到最后，只在选好模型后评估一次。这样可以避免把观察过的测试结果反复用于调参。分类树在叶子中通常预测多数类别，也可用训练样本的类别比例估计概率。

下面从树图顶部往下读：左分支表示条件成立，右分支表示条件不成立。找一个叶子，试着说出到达它需要满足哪些条件。

In [ ]:
rng = np.random.default_rng(24)
n = 600
advertising = rng.uniform(0, 10, n)
event = rng.integers(0, 2, n)
weather = rng.uniform(0, 10, n)  # 数值越高天气越好
X = np.column_stack([advertising, event, weather])
# 有随机波动，因此即使条件相同，结果也不一定一样。
score = .65 * advertising + 2.0 * event + .35 * weather + rng.normal(0, 2.0, n)
y_class = (score > 5.7).astype(int)
y_count = 20 + 2.2 * advertising + 8 * event + 1.4 * weather + 4 * (advertising > 5) + rng.normal(0, 4, n)
# 先留出测试集，再从剩余数据中分出验证集。
X_pool, X_test, y_pool, y_test, count_pool, count_test = train_test_split(
    X, y_class, y_count, test_size=.2, random_state=42, stratify=y_class)
X_train, X_valid, y_train, y_valid, count_train, count_valid = train_test_split(
    X_pool, y_pool, count_pool, test_size=.25, random_state=11, stratify=y_pool)

small_tree = DecisionTreeClassifier(max_depth=2, random_state=42).fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(small_tree, feature_names=['Advertising', 'Event', 'Weather'],
          class_names=['No', 'Yes'], filled=True, rounded=True, ax=ax)
plt.show()
print(f'训练准确率 {accuracy_score(y_train, small_tree.predict(X_train)):.3f}；验证准确率 {accuracy_score(y_valid, small_tree.predict(X_valid)):.3f}')
print('新一天 [宣传=6, 有活动=1, 天气=5] 的售罄概率：', round(small_tree.predict_proba([[6, 1, 5]])[0, 1], 3))

图中 `samples` 是该节点的训练样本数，`value` 表示两类在该节点的分布（不同版本可能显示数量或比例），`gini` 衡量混杂程度。样本沿条件走到叶子；不是每个日期都经过所有问题。概率来自叶子里的比例，不代表一定售罄，也不保证概率已经校准。

## 3. 如果树一直长下去，会发生什么？

一棵深树可以把训练日分得越来越细，甚至记住偶然噪声；训练表现变好，不代表新数据也更好。先猜：深度不断增加时，训练集和验证集的准确率会怎样变化？然后拖动滑块。

限制深度是**预剪枝**；也可以先长树，再用 `ccp_alpha` 进行代价复杂度**后剪枝**。约束叶子的最少样本数也能减少少数样本对预测的影响。验证集供本章选择和观察，独立测试集暂时不参与。

In [ ]:
# 【演示】每个深度都从同一份训练集重新训练。
def show_depth(depth):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42).fit(X_train, y_train)
    train_acc = accuracy_score(y_train, tree.predict(X_train))
    valid_acc = accuracy_score(y_valid, tree.predict(X_valid))
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.bar(['Train', 'Validation'], [train_acc, valid_acc], color=['#4169a1', '#e88b36'])
    ax.set(ylim=(0, 1.05), ylabel='Accuracy', title=f'Max depth = {depth}; leaves = {tree.get_n_leaves()}')
    for i, v in enumerate([train_acc, valid_acc]): ax.text(i, v + .02, f'{v:.3f}', ha='center')
    plt.show()

display(widgets.interactive(show_depth, depth=widgets.IntSlider(value=2, min=1, max=15, description='Depth', continuous_update=False)))

In [ ]:
# 【练习】比较未剪枝和后剪枝；修改 ccp_alpha 试试。
full_tree = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
pruned_tree = DecisionTreeClassifier(ccp_alpha=.01, random_state=42).fit(X_train, y_train)
for name, tree in [('Full', full_tree), ('Pruned', pruned_tree)]:
    print(f'{name:6s} 叶子 {tree.get_n_leaves():3d}  训练 {tree.score(X_train, y_train):.3f}  验证 {tree.score(X_valid, y_valid):.3f}')

一次随机划分只能提供一组观察，验证结果也会波动。实际选深度、剪枝和其他超参数时，可在训练数据内使用交叉验证；把测试集留到模型选择完成后。即使控制了深度，单棵树仍可能对训练样本的变化很敏感：能不能把多棵不同的树组合起来，让结果更稳定？

### 自测 2：判断过拟合

结合刚才的训练与验证表现，选择最合理的解释。

In [ ]:
# 【自测】选择后提交，可修改答案重试。
display(lesson_quiz('overfit'))

## 4. 从一棵树到多棵树：随机森林

先想象对同一问题分别请多位观察者判断。如果每位都看见完全相同的数据，并总用同样的方法，他们可能重复同样的错误。要让组合有意义，需要让各棵树看到有所不同的数据或候选特征。

### 先认识 Bagging，再看随机森林

![对比示意：Bagging 与随机森林把不同训练样本送给各棵独立树并合并，梯度提升树让后一棵树基于前面结果逐轮修正](../assets/forest_vs_boosting.png)

图的左侧先看随机森林；右侧 GBDT 是后面将学习的另一种合作方式，暂时不必理解。

**Bagging**（bootstrap aggregating）对训练集有放回抽样，得到多个大小相同的训练样本集；同一行可能被抽到多次，也可能一次未被抽到。各树分别训练，最后分类投票或回归取平均。**随机森林在此基础上**，在每次分裂时还随机抽取部分特征作为候选，使树不要总被同一强特征主导。图中的并行表示树之间不依赖前一棵树，不代表所有实现或硬件一定同时执行。

想一想：若每棵树完全一样，平均后能消除它们共同犯的错吗？多样性有助于降低波动，但各树之间仍可能相关；它并不能保证模型无偏，也不能保证森林总比单棵树好。

In [ ]:
forest = RandomForestClassifier(n_estimators=100, min_samples_leaf=3, random_state=42, n_jobs=-1)
forest.fit(X_train, y_train)
example = np.array([[6, 1, 5]])
votes = np.array([tree.predict(example)[0] for tree in forest.estimators_])
print(f'100 棵树中预测售罄的票数：{int(votes.sum())}；其余：{len(votes)-int(votes.sum())}')
print(f'单棵深树验证准确率：{full_tree.score(X_valid, y_valid):.3f}')
print(f'随机森林验证准确率：{forest.score(X_valid, y_valid):.3f}')
print(f'森林预测售罄概率：{forest.predict_proba(example)[0, 1]:.3f}')

这里每棵树的 `predict` 给出一票；scikit-learn 分类森林的 `predict_proba` 实际是平均各树的叶子类别概率，再按平均概率选类别，因此不必恰好等于硬投票比例。森林通常比单棵树难以完整解释，特征重要性也不等同于因果关系。

### 自测 3：随机森林的多样性

思考森林中的树如何训练，以及为什么它们不会完全相同。

In [ ]:
# 【自测】选择后提交，可修改答案重试。
display(lesson_quiz('forest'))

### 回看：树的两条组合路线

![学习地图：决策树可以用于 Bagging 与随机森林，也可以用于 Boosting 与梯度提升树；XGBoost 的树模型属于后一条路线](../assets/tree_learning_map.png)

现在已了解单棵树和随机森林，可以沿图从左向右继续：**Bagging → 随机森林**让不同树分别学习，再合并结果；接下来要学的 **Boosting → GBDT → XGBoost 树模型**则让后一轮利用已有模型的结果。AdaBoost 是另一类提升方法，这里只标出它的位置。

箭头表示概念关系，不是说随机森林训练之后会变成 GBDT。要理解下一条路线，先将目标从“是否售罄”换成连续的“销售多少个”：这样更容易亲手计算模型还差多少。

## 5. 回归树：预测面包数量，为逐轮修正做准备

之前的分类树预测“售罄还是未售罄”，现在我们预测连续销量。**回归树**常用叶子内训练目标的平均值作为输出，并选择使两边平方误差总和下降的划分。它的预测形成一段段平坦台阶，而不是线性回归的直线；普通回归树难以外推到训练目标范围之外。

先观察下面的台阶：相同区间内的不同广告投入，为什么会得到相同预测？理解这个回归叶子之后，我们就可以让下一棵树预测模型的误差。

In [ ]:
# 为看清台阶，只用宣传这一个输入来预测数量。
reg_tree = DecisionTreeRegressor(max_depth=3, random_state=42).fit(X_train[:, [0]], count_train)
grid = np.linspace(0, 10, 200).reshape(-1, 1)
plt.figure(figsize=(8, 4))
plt.scatter(X_train[:, 0], count_train, s=12, alpha=.25, label='Train days')
plt.plot(grid[:, 0], reg_tree.predict(grid), color='#e88b36', lw=3, label='Tree prediction')
plt.xlabel('Advertising (hundreds)'); plt.ylabel('Bread sold'); plt.legend(); plt.grid(alpha=.2); plt.show()
print(f'验证均方误差：{mean_squared_error(count_valid, reg_tree.predict(X_valid[:, [0]])):.2f}')

注意：这个演示只用了宣传，忽略了活动与天气，不能与后面的三特征模型直接比较优劣。均方误差越小，说明这批样本的数量预测整体越接近实际，但单位是数量的平方。

## 6. 梯度提升树：一棵接一棵修正

随机森林分别训练多棵树，再平均或合并；现在换一个问题：如果已有销量预测偏高或偏低，能否训练一棵新树专门学习这个偏差，然后把修正加到原预测上？这就是梯度提升的入手方式。

以**平方误差回归**为例，先用训练销量的平均值作初始预测，计算每个训练样本的残差 $r_i=y_i-\hat y_i$；新树拟合残差，乘学习率 $\eta$ 后加到原预测上：

$$\hat y^{(m)}=\hat y^{(m-1)}+\eta\,h_m(x)$$

更新后要重新计算残差，再训练下一棵树；每棵新树学习的是**修正量**，不是单独预测总销量。更一般地，梯度提升让新树拟合损失函数的负梯度，分类或其他损失不一定是普通的 $y-\hat y$ 残差。

![梯度提升示意：初始预测与真实值产生残差，第一棵树拟合残差并按学习率修正，更新后的残差供下一轮使用](../assets/gbdt_residuals.png)

**为什么说它是“提升”？** 把当前多棵树的总输出视作一个模型：新树在训练集上看到当前模型的误差信息，学习一个修正量，然后加入总预测。下一轮先重新计算误差信息，因此树的顺序有意义。随机森林中的树大体是并列的、输出再平均；梯度提升中的树是顺序依赖的、输出逐项相加。

这里的残差解释专指**平方误差回归**。例如真实销量为 12、当前预测为 8，残差为 4；若下一棵树对该样本输出 3、学习率为 0.2，新预测为 $8+0.2\times 3=8.6$，剩余残差为 3.4。树对所有训练样本共同拟合，不保证每个样本每一轮都更准；不同损失则拟合相应的负梯度，不能一概当作销量残差。

In [ ]:
# 【演示】逐轮观察提升树：先训练一次，再查看每轮的预测与误差。
boost = GradientBoostingRegressor(n_estimators=100, learning_rate=.08, max_depth=2, random_state=42)
boost.fit(X_train, count_train)
train_stages = list(boost.staged_predict(X_train))
valid_stages = list(boost.staged_predict(X_valid))
train_mse = [mean_squared_error(count_train, pred) for pred in train_stages]
valid_mse = [mean_squared_error(count_valid, pred) for pred in valid_stages]

def show_boost(stage):
    fig, (ax, line) = plt.subplots(1, 2, figsize=(10, 3.5))
    ax.scatter(count_valid, valid_stages[stage-1], alpha=.45, s=18, color='#4169a1')
    lo = min(count_valid.min(), valid_stages[stage-1].min())
    hi = max(count_valid.max(), valid_stages[stage-1].max())
    ax.plot([lo, hi], [lo, hi], '--', color='#e88b36')
    ax.set(xlabel='Actual bread sold', ylabel='Prediction', title=f'Validation predictions: tree {stage}')
    line.plot(range(1, 101), train_mse, label='Train', color='#4169a1')
    line.plot(range(1, 101), valid_mse, label='Validation', color='#e88b36')
    line.axvline(stage, color='#d33f49', linestyle='--')
    line.set(xlabel='Number of trees', ylabel='MSE', title='Error by stage'); line.legend()
    for a in (ax, line): a.grid(alpha=.2)
    plt.tight_layout(); plt.show()

display(widgets.interactive(show_boost, stage=widgets.IntSlider(value=1, min=1, max=100, description='Trees', continuous_update=False)))

拖动树的数量，观察预测点、训练误差和验证误差的变化。树的数量、深度和学习率都影响结果：提升树也可能过拟合，验证误差不保证每轮都下降。实际选择轮数时只看验证结果，最后再用测试集评估。下面可以选做一个从零实现；它与后面的 XGBoost 小节互不依赖，跳过不影响继续学习。

### 自测 4：做一次提升更新

先手算修正量，再选择更新后的预测。

In [ ]:
# 【自测】选择后提交，可修改答案重试。
display(lesson_quiz('boost'))

### 选学：只用 NumPy 写一次回归树划分与提升

刚才看到了平方误差下的逐轮修正。这里先写一个只分裂一次的回归树（**树桩**），再将这些树桩依次相加，亲手重现这一过程。两格都可跳过；后续课程不依赖它们。

这只是机制演示，没有多特征搜索、缺失值处理或自动调参。第一个函数遍历相邻输入的中点，选择平方误差最小的门槛；两个叶子分别输出对应目标的平均数。

In [ ]:
# 【算法练习】在相邻不同输入的中点中，寻找平方误差最小的划分。
def fit_stump(x, target):
    x, target = np.asarray(x, dtype=float), np.asarray(target, dtype=float)
    unique = np.unique(x)
    if len(unique) < 2:
        raise ValueError('至少需要两个不同的输入值')
    thresholds = (unique[:-1] + unique[1:]) / 2
    best = None
    for threshold in thresholds:
        left = x <= threshold
        left_mean, right_mean = target[left].mean(), target[~left].mean()
        loss = np.sum((target[left] - left_mean)**2) + np.sum((target[~left] - right_mean)**2)
        if best is None or loss < best[0]:
            best = (loss, threshold, left_mean, right_mean)
    return best[1:]

def predict_stump(x, stump):
    threshold, left_mean, right_mean = stump
    return np.where(np.asarray(x) <= threshold, left_mean, right_mean)

x_toy = np.array([0., 1., 2., 3., 4., 5.])
y_toy = np.array([2., 2., 3., 7., 8., 8.])
stump = fit_stump(x_toy, y_toy)
np.testing.assert_allclose(stump, (2.5, 7/3, 23/3))
print('门槛、左叶平均、右叶平均：', stump)

In [ ]:
# 【算法练习】每轮拟合当前残差，再将树桩输出乘学习率加回预测。
def fit_boost_numpy(x, y, rounds=8, learning_rate=.3):
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    base = float(y.mean())
    prediction = np.full(len(y), base)
    stumps, losses = [], [np.mean((y - prediction)**2)]
    for _ in range(rounds):
        residual = y - prediction
        stump = fit_stump(x, residual)
        stumps.append(stump)
        prediction += learning_rate * predict_stump(x, stump)
        losses.append(np.mean((y - prediction)**2))
    return base, stumps, np.array(losses)

def predict_boost_numpy(x, base, stumps, learning_rate):
    prediction = np.full(len(x), base, dtype=float)
    for stump in stumps:
        prediction += learning_rate * predict_stump(x, stump)
    return prediction

rate = .3
base, stumps, losses = fit_boost_numpy(x_toy, y_toy, rounds=8, learning_rate=rate)
first_residual = y_toy - y_toy.mean()
np.testing.assert_allclose(stumps[0], fit_stump(x_toy, first_residual))
# 固定学习率验证例子；实验学习率过大时，误差可能上升。
_, _, check_losses = fit_boost_numpy(x_toy, y_toy, rounds=8, learning_rate=.3)
assert np.all(np.diff(check_losses) <= 1e-10)
print('初始预测：', base, '；每轮训练 MSE：', np.round(losses, 3))
print('最终预测：', np.round(predict_boost_numpy(x_toy, base, stumps, rate), 2))

## 7. XGBoost：在梯度提升上怎样选择下一棵树？

我们已经看到普通梯度提升怎样根据误差逐轮修正。接下来保留这一框架，看看 XGBoost 的树模型如何利用损失的导数和复杂度惩罚来计算新树。

![XGBoost 示意：由当前预测与标签计算一阶梯度和二阶导数，汇总到候选叶子，比较考虑复杂度惩罚的分裂增益，再把新树按学习率加入模型](../assets/xgboost_mechanics.png)

XGBoost 的默认树 booster 沿用“逐轮加树”的框架，但构建一棵新树时，可以对当前预测附近的损失作**二阶近似**：每条训练记录提供一阶导数 $g_i$（损失随预测变化的方向）和二阶导数 $h_i$（局部曲率）。把属于同一叶子的 $g_i$、$h_i$ 分别相加，得到 $G$、$H$；模型会综合考虑拟合改善和树的复杂度。这里以常见的可微目标、L2 叶权正则、没有 L1 正则为例：

$$w^*=-\frac{G}{H+\lambda}$$

$$\text{分裂增益}=\tfrac12\left[\frac{G_L^2}{H_L+\lambda}+\frac{G_R^2}{H_R+\lambda}-\frac{G^2}{H+\lambda}\right]-\gamma$$

$w^*$ 是近似目标下该叶子的最优输出；$\lambda$ 惩罚较大的叶权，$\gamma$ 是增加一个叶子的复杂度代价。增益比较的是一次分裂前后近似目标的改善，实际是否分裂还受实现的约束和参数影响。分类时这些导数针对模型的**原始分数**计算，而不是把概率直接当作树的输出。

下面做一个可手算例子：损失取 $\tfrac12(y-\hat y)^2$，当前预测都是 5，$g_i=\hat y_i-y_i$，$h_i=1$。先分别求两片叶子的权重，再比较拆分前后的增益；随后才把树输出乘学习率加入预测。

In [ ]:
# 【算法练习】XGBoost 风格的一次分裂：平方误差、仅 L2 正则。
y_example = np.array([2., 4., 6., 8.])
pred_before = np.full(4, 5.)
g = pred_before - y_example
h = np.ones_like(g)
lambda_l2, gamma = 1., 0.

def leaf_weight(g_values, h_values, l2):
    return -np.sum(g_values) / (np.sum(h_values) + l2)

def split_gain(g_left, h_left, g_right, h_right, l2, gamma):
    gl, hl = np.sum(g_left), np.sum(h_left)
    gr, hr = np.sum(g_right), np.sum(h_right)
    parent = (gl + gr)**2 / (hl + hr + l2)
    return .5 * (gl**2 / (hl + l2) + gr**2 / (hr + l2) - parent) - gamma

w_left = leaf_weight(g[:2], h[:2], lambda_l2)
w_right = leaf_weight(g[2:], h[2:], lambda_l2)
gain = split_gain(g[:2], h[:2], g[2:], h[2:], lambda_l2, gamma)
np.testing.assert_allclose(g, [3., 1., -1., -3.])
np.testing.assert_allclose(h, [1., 1., 1., 1.])
np.testing.assert_allclose([leaf_weight(g[:2], h[:2], 1.),
                            leaf_weight(g[2:], h[2:], 1.),
                            split_gain(g[:2], h[:2], g[2:], h[2:], 1., 0.)],
                           [-4/3, 4/3, 16/3])
print('梯度:', g, '；二阶导数:', h)
print('左、右叶权:', round(w_left, 3), round(w_right, 3), '；分裂增益:', round(gain, 3))
# 试着提高 lambda_l2 或 gamma，观察叶权和增益怎么变化；上面的检查固定使用 1 和 0。

二阶近似、正则化、采样、缺失值处理或直方图搜索**并非 XGBoost 独有**；不同梯度提升实现也可能提供其中一些能力。XGBoost 的树 booster 将这些机制组合在可配置的实现中。它可以并行计算同一轮内的部分工作，但**后一轮依赖前一轮的预测**，不是把各轮树当作独立森林同时训练。

参考：[XGBoost 官方模型教程](https://xgboost.readthedocs.io/en/stable/tutorials/model.html)；[scikit-learn 集成方法指南](https://scikit-learn.org/stable/modules/ensemble.html)。

### 自测 5：理解正则化

用上一节的叶权公式，想一想参数改变会产生什么影响。

In [ ]:
# 【自测】选择后提交，可修改答案重试。
display(lesson_quiz('xgb'))

## 8. 在同一道题上比较，并留出最终评估

前面分类树与森林预测“是否售罄”，回归树与 GBDT 预测“销量”，准确率和均方误差不能直接比较。现在统一为**销量回归**：各模型使用相同的三个特征、相同训练集、同一个验证集和均方误差；参数预先固定。下面的结果供选择这组候选模型，并不证明某一种算法永远更好。

先在验证集比较。若已安装 XGBoost，也可把它作为候选；确定候选后再用未参与观察的测试集评估选中模型一次。实际项目中，还可以用交叉验证调参；按时间发生的数据应考虑时间划分。

In [ ]:
# 【演示】相同回归任务、相同训练/验证划分、相同指标。
X_fit, count_fit = X_train, count_train
regressors = {
    'Decision tree': DecisionTreeRegressor(max_depth=4, random_state=42),
    'Random forest': RandomForestRegressor(n_estimators=100, min_samples_leaf=3, random_state=42, n_jobs=-1),
    'Gradient boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=.08, max_depth=2, random_state=42),
}
validation_scores = {}
for name, estimator in regressors.items():
    estimator.fit(X_fit, count_fit)
    validation_scores[name] = mean_squared_error(count_valid, estimator.predict(X_valid))
    print(f'{name:18s} 验证 MSE = {validation_scores[name]:.2f}')

### 可选：使用 XGBoost 库做同一回归任务

本课不要求安装 XGBoost；如当前环境已安装，下面才训练它并在同一个验证集上计算 MSE，把它加入候选模型。不同库的参数并非一一等价；为了公平评价，需要在相同的数据和验证规则下分别调参，不能只凭这组默认示例断定谁最好。

In [ ]:
# 【可选】无 xgboost 时跳过，前面的课程仍可完整运行。
import importlib.util
if importlib.util.find_spec('xgboost') is None:
    print('未安装 xgboost，跳过可选对照。')
else:
    from xgboost import XGBRegressor
    xgb_model = XGBRegressor(n_estimators=100, max_depth=2, learning_rate=.08,
                             objective='reg:squarederror', random_state=42, n_jobs=1)
    xgb_model.fit(X_fit, count_fit)
    regressors['XGBoost'] = xgb_model
    validation_scores['XGBoost'] = mean_squared_error(count_valid, xgb_model.predict(X_valid))
    print(f"XGBoost 验证 MSE = {validation_scores['XGBoost']:.2f}")

### 选好模型后，才打开测试集

下面根据上面的验证均方误差选择一个已训练的候选模型，再对从未用于学习或选择的测试集计算一次均方误差。测试结果用来估计这次选择的泛化表现，不应用它反复更换模型或参数。若重新尝试方案，应回到训练／验证过程，并保留新的独立数据作最终评估。

In [ ]:
best_name = min(validation_scores, key=validation_scores.get)
best_model = regressors[best_name]
test_mse = mean_squared_error(count_test, best_model.predict(X_test))
print(f'验证集选出的模型：{best_name}')
print(f'最终测试 MSE：{test_mse:.2f}')

### 回顾：把几种方法放在一起

| 问题 | 单棵决策树 | 随机森林 | GBDT / XGBoost 的树 booster |
| --- | --- | --- | --- |
| 如何训练 | 递归选择划分 | 各树独立拟合重采样数据，分裂时随机选候选特征 | 下一棵树依赖当前模型，逐轮降低目标 |
| 如何合并 | 一条路径到一个叶子 | 分类合并各树结果，回归平均 | 各轮修正量加到已有预测上 |
| 着重解决 | 用规则拟合非线性关系 | 借助多样化与平均减小预测波动 | 用一系列修正逐步改善拟合 |
| 需要留意 | 深树容易过拟合 | 树多也不能修复系统性偏差 | 学习率、轮数和树复杂度共同影响过拟合 |

树的数量、深度、特征采样和数据质量都会影响结果；模型复杂不等于一定更准确。

### 自测 6：综合检查

比较模型后，怎样保留一次可信的最终评估？

In [ ]:
# 【自测】选择后提交，可修改答案重试。
display(lesson_quiz('evaluation'))

## 带走什么？

- **决策树**：逐次提问，比较划分后的纯度或误差，把样本送到叶子；树太深可能拟合噪声。
- **Bagging / 随机森林**：各树从不同的样本学习，森林再随机选候选特征；通过投票或平均组合，树之间不依赖训练顺序。
- **GBDT**：依次增加树，当前模型的预测决定下一轮的修正方向；平方误差回归中负梯度与残差相同或只差比例，其他损失要看对应的负梯度。
- **XGBoost 的树 booster**：属于梯度提升树的实现；按目标函数的一阶、二阶信息与复杂度惩罚构造新树。XGBoost 是库，不能说所有 booster 都是树。

**挑战**：① 手算门槛 2.5 两边的加权基尼，并与滑块对照；② 解释随机森林与梯度提升树分别怎样利用多棵树；③ 修改 NumPy 提升的学习率和轮数，画出 `losses`；④ 在上面的叶权例子中提高 `lambda_l2` 或 `gamma`，预测权重与分裂增益如何变化，再运行验证；⑤ 为什么同一测试集不能反复用来挑选模型？